In [1]:
import os
os.environ["OPENAPI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [2]:
!pip install --upgrade youtube-transcript-api langchain-community langchain-openai langchain-text-splitters faiss-cpu tiktoken python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

d:\Data-Science-Gen-AI-Projects\Gen-AI-Projects\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 1a: Indexing (Document Ingestion)

In [11]:
video_id = "kFGCyVTAn50"  # Example YouTube video ID
try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript[:500])  # Print the first 500 characters of the transcript
except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")

Hello all, my name is Krishna and welcome to my YouTube channel. So guys, I am super excited to announce a new live boot camp which focuses specifically for industry professionals wherein we are going to learn about how to learn AI by using modern route. Now in this specific batch we are going to focus on full stack generative AI and agentic AI applications wherein we are not not going to discuss about how to create applications using different kind of LLMs but instead we focus on more on fine-t


In [12]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Hello all, my name is Krishna and', start=0.64, duration=5.119), FetchedTranscriptSnippet(text='welcome to my YouTube channel. So guys,', start=2.96, duration=5.92), FetchedTranscriptSnippet(text='I am super excited to announce a new', start=5.759, duration=6.321), FetchedTranscriptSnippet(text='live boot camp which focuses', start=8.88, duration=6.08), FetchedTranscriptSnippet(text='specifically for industry professionals', start=12.08, duration=5.279), FetchedTranscriptSnippet(text='wherein we are going to learn about how', start=14.96, duration=6.159), FetchedTranscriptSnippet(text='to learn AI by using modern route. Now', start=17.359, duration=5.521), FetchedTranscriptSnippet(text='in this specific batch we are going to', start=21.119, duration=3.841), FetchedTranscriptSnippet(text='focus on full stack generative AI and', start=22.88, duration=5.44), FetchedTranscriptSnippet(text='agentic AI applications wherein we are', s

### Step 1b: Indexing (Text Splitting)

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [14]:
len(chunks)

16

In [15]:
chunks[0]

Document(metadata={}, page_content="Hello all, my name is Krishna and welcome to my YouTube channel. So guys, I am super excited to announce a new live boot camp which focuses specifically for industry professionals wherein we are going to learn about how to learn AI by using modern route. Now in this specific batch we are going to focus on full stack generative AI and agentic AI applications wherein we are not not going to discuss about how to create applications using different kind of LLMs but instead we focus on more on fine-tuning we focus on building AI agents from scratch and even solve complex workflows which are specifically required in the industries. So more about this particular batch please make sure you watch this uh video till the end because I'm also going to give you a specific road map like how you should probably use modern route in order to learn AI very much important for industry professionals okay who are having couple of years of experience who know Python progr

### Step 1c & 1d: Indexing (Embedding Generation and Storing in Vector Store)

In [16]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

In [17]:
vector_store.index_to_docstore_id

{0: '18346731-2874-4382-890b-24b32b2b1cb3',
 1: '9c261bde-35b8-489a-a988-d4f66920f068',
 2: '04544e02-2063-4f51-a9c6-36df99f34b3c',
 3: 'df719a12-87b0-4e96-b16c-b6c13819fb84',
 4: '0300d0ad-b8cd-40af-b805-887724bf6606',
 5: '45029588-2c2f-470f-80d2-42995d570eeb',
 6: 'a4d4e8b5-4aab-40c1-94ed-0a8ca5cecd26',
 7: 'fac69390-6e15-4455-9e19-8672415def6d',
 8: '09a47a17-1d32-4e21-857e-5212bc5cfbc3',
 9: '4f2ad00a-4968-4dbe-a46f-80478d6ae2bb',
 10: 'e4df35c1-c516-45e6-9745-c31918260d62',
 11: '6e29e6d7-4622-4339-9c69-7fdc59533d5d',
 12: '26d54ee7-1134-41a4-8611-4dd5764d6c01',
 13: 'd6c4aff0-1971-46b1-a1d2-beb0ba35bc34',
 14: 'f47e61db-bc42-40d0-b5d1-bed790b4faec',
 15: 'ef57994a-589a-4636-9369-78e76436f25d'}

In [19]:
vector_store.get_by_ids(['f47e61db-bc42-40d0-b5d1-bed790b4faec'])

[Document(id='f47e61db-bc42-40d0-b5d1-bed790b4faec', metadata={}, page_content="counseling team number if you have any queries whom to probably join even though I've explained it perfectly over here. If you want to quickly get started into generative AI and agent you have basic knowledge of machine learning, deep learning, Python programming language prerequisite Python is specifically required I think you should be able to go ahead and start this specific batch. Okay. And again as said uh this is one of a type of a batch only in a year we probably launched this flagship batches like uh ultimate data science like modern route for that was for the traditional route. This is for the modern route and yes for the advanced route we will be coming up with more specialization in the future. Uh but we don't launch batch regularly. We focus more on teaching you in a way that it impacts your career. Right? So I hope you like this particular video. Uh I think I've covered almost each and everythi

### Step 2: Retrieval

In [20]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [21]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000019C2573A030>, search_kwargs={'k': 4})

In [22]:
retriever.invoke('What is deep agents?')

[Document(id='45029588-2c2f-470f-80d2-42995d570eeb', metadata={}, page_content="ahead and add agentic AI skill sets. We start building agents. We covered different different frameworks like lang chain, lang graph, different different frameworks we have also included in the our syllabus which I will be showing you. Okay. And then parallelly you can also go ahead and learn DS fundamentals. Now in this particular batch that the announcement that I'm actually making is for this particular path uh that is nothing but full stack genai and agentic AI boot camp. Okay. So this entire course is basically targeting the modern route. So in Krishna Academy we are going to only follow this three parts. U the third part which is the advanced route. Here we are going to introduce some specialized course right. Specialized course basically means like you want to only focus on you know agentic AI building AI agents. You want to probably go ahead and focus on building rags right? So like rags like agenti

### Step 3: Augmentation

In [23]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature = 0.2)

In [25]:
prompt = PromptTemplate(
    template = """
You are a helpful assistant.
Answer only from the provided transcript context.
if the context is insuficient, just say you don't know.

{context}
Question: {question}
""", 
input_variables=["context", "question"]
)

In [26]:
question = "Is the topic of agents discussed in the video? If yes, what are the key points mentioned about agents? "
retrieved_docs = retriever.invoke(question)

In [27]:
retrieved_docs

[Document(id='a4d4e8b5-4aab-40c1-94ed-0a8ca5cecd26', metadata={}, page_content='agentic AI building AI agents. You want to probably go ahead and focus on building rags right? So like rags like agentic AI AI agents right? Uh where we cover almost uh everything with respect to advanced topics. we will be covering in this advanced route. But right now our target is basically to start one batch in March 15th. And again here you will be able to see that this two dedicated batch will be once in a year right when we talk about this one and we talk about this one right and then we only focus on advanced batches which will be coming coming up in the future and this particular batch will be only focused for people who want to do advanced learning in the specific field uh like rag like AI agents and all but here also we are covering AI agents we are covering almost everything in Gen AI agentic AI We are covering multiple frameworks and then we also cover genai completely from basics where from wh

In [28]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"agentic AI building AI agents. You want to probably go ahead and focus on building rags right? So like rags like agentic AI AI agents right? Uh where we cover almost uh everything with respect to advanced topics. we will be covering in this advanced route. But right now our target is basically to start one batch in March 15th. And again here you will be able to see that this two dedicated batch will be once in a year right when we talk about this one and we talk about this one right and then we only focus on advanced batches which will be coming coming up in the future and this particular batch will be only focused for people who want to do advanced learning in the specific field uh like rag like AI agents and all but here also we are covering AI agents we are covering almost everything in Gen AI agentic AI We are covering multiple frameworks and then we also cover genai completely from basics where from where we start like transformers we understand the architecture of transformers\n

In [29]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

### Step 4: Generation

In [30]:
answer = llm.invoke(final_prompt)
answer

AIMessage(content='Yes, the topic of agents is discussed in the video. The key points mentioned about agents include:\n\n1. The focus on building agentic AI and AI agents, particularly in the context of advanced learning.\n2. The course covers various frameworks for building agents, such as Lang Chain and Lang Graph.\n3. The curriculum includes advanced topics like retrieval augmented generation (RAG), multi-agent systems, and deep agent systems.\n4. The importance of agents in solving complex workflows and their applications in B2B and B2C products.\n5. The course aims to provide a comprehensive understanding of agentic AI and its practical applications in industries.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 847, 'total_tokens': 975, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens'

### Building the Chain

In [31]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [32]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [33]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [34]:
parallel_chain.invoke('What is deep agents?')

{'context': "ahead and add agentic AI skill sets. We start building agents. We covered different different frameworks like lang chain, lang graph, different different frameworks we have also included in the our syllabus which I will be showing you. Okay. And then parallelly you can also go ahead and learn DS fundamentals. Now in this particular batch that the announcement that I'm actually making is for this particular path uh that is nothing but full stack genai and agentic AI boot camp. Okay. So this entire course is basically targeting the modern route. So in Krishna Academy we are going to only follow this three parts. U the third part which is the advanced route. Here we are going to introduce some specialized course right. Specialized course basically means like you want to only focus on you know agentic AI building AI agents. You want to probably go ahead and focus on building rags right? So like rags like agentic AI AI agents right? Uh where we cover almost uh everything with r

In [35]:
parser = StrOutputParser()

In [36]:
main_chain = parallel_chain | prompt | llm | parser

In [37]:
main_chain.invoke('Can you summarize the video?')

'The video discusses a course on generative AI and agent technology, highlighting its syllabus and structure. It mentions that the course is designed for individuals with basic knowledge of machine learning, deep learning, and Python programming. The course will last for 6 months and includes access to a dashboard for 1.5 years. It covers a wide range of topics, including transformer architecture, text encoding, fine-tuning techniques, and various cloud services. The instructor emphasizes the importance of the course in impacting careers and mentions that it is a flagship batch offered only once a year. The video encourages viewers to check the syllabus and get started with enrollment.'